# Explore the contributions from MaLAPA 2026

Reads the cleaned data from **step 1** (`step_1_output/contributions_rewritten.json`)
and produces the figures. **Run `step_1_prepare_data.ipynb` first.**

Outputs, written into `step_2_output/`: four PNG snapshots of the plots and
`institution_submissions_by_community.txt`.

### How to read the charts
- Each contribution is attributed to its **first speaker's** institution (a
  multi-speaker contribution counts only toward the first speaker).
- **Species** = the kind of particle the institution's accelerator work centers
  on (electron vs hadron machines), assigned one value per institution.
- The **collaboration network** links two institutions whenever they appear
  together on a contribution, weighted by the number of contributions they share.
- **One institution per person.** Where an individual listed more than one
  institution, only one was chosen, so each person counts toward a single
  institution.


In [ ]:
import json
from pathlib import Path
from pprint import pprint
from malapa_helpers import iter_people, first_speaker
from collections import Counter
from itertools import combinations
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pycountry
import networkx as nx

### Load the cleaned data (from step 1)

In [ ]:
JSON_FILE_PATH = Path("step_1_output/contributions_rewritten.json")
with open(JSON_FILE_PATH, "r") as f:
    data = json.load(f)

### Headline numbers

In [ ]:
total_contributions = len(data)
poster_contributions = sum(
    1 for contribution in data if str(contribution.get("type", "")).strip().lower() == "poster"
)

first_speaker_countries = {
    fs.get("country")
    for contribution in data
    if (fs := first_speaker(contribution)) and fs.get("country")
}

print(f"Total contributions: {total_contributions}")
print(f"Poster contributions: {poster_contributions}")
print(f"Countries represented by first speaker: {len(first_speaker_countries)}")

Total contributions: 94
Poster contributions: 62
Countries represented by first speaker: 11


### Load institution locations (for the map)

In [ ]:
JSON_FILE_PATH = Path("source_data/institution_locations.json")
with open(JSON_FILE_PATH, "r") as f:
    locations = json.load(f)

### Helper — save each figure to a PNG

In [ ]:
output_dir = Path("step_2_output")
output_dir.mkdir(parents=True, exist_ok=True)

# Save each Plotly figure below into step_2_output/ as a PNG.
# (Plotly's native write_image renders via kaleido under the hood.)
def save_fig(fig, filename, scale=2):
    path = output_dir / filename
    fig.write_image(path, scale=scale)
    print(f"Saved plot to: {path}")

## Contributions by institution (bar chart)

Contributions per institution (by first speaker), colored by species.

In [ ]:
location_counts = {location["inst"]: 0 for location in locations}

for contribution in data:
    fs = first_speaker(contribution)
    if fs is None:
        continue
    first_speaker_inst = fs.get("inst")
    if first_speaker_inst in location_counts:
        location_counts[first_speaker_inst] += 1

counts_df = pd.DataFrame(
    [
        {
            "inst": location["inst"],
            "label": location.get("label_short", location["inst"]),
            "count": location_counts[location["inst"]],
            "country": location.get("country"),
            "species": location.get("species"),
        }
        for location in locations
    ]
)
counts_df = counts_df[counts_df["count"] > 0].sort_values("count", ascending=False)

display(counts_df)

fig = px.bar(
    counts_df,
    x="inst",
    y="count",
    hover_name="label",
    hover_data={"country": True, "species": True, "inst": False, "count": True},
    color="species",
    title="Contributions by First Speaker Institution",
    labels={"inst": "Institution", "count": "Contributions", "species": "Species"},
)
fig.update_layout(xaxis_tickangle=-45)
save_fig(fig, "Contributions_by_Institution.png")
fig.show()

,inst,label,count,country,species
8,DESY,"DESY, Hamburg, DEU",14,Germany,electrons
38,CERN,"CERN, Geneva, CHE",13,Switzerland,hadrons
32,BNL,"BNL, Upton, USA",9,United States,hadrons
30,LBNL,"LBNL, Berkeley, USA",7,United States,electrons
2,FNAL,"FNAL, Batavia, USA",7,United States,hadrons
5,SLAC,"SLAC, Menlo Park, USA",7,United States,electrons
40,ISIS,"ISIS, Didcot, GBR",5,United Kingdom,hadrons
46,PAL,"PAL, Pohang, KOR",4,South Korea,electrons
14,JLab,"JLab, Newport News, USA",3,United States,electrons
45,RSC,"RSC, Sayo, JPN",2,Japan,electrons


/var/folders/n0/04pzpmy918x8zphf4zjqct2c0000j6/T/ipykernel_64766/1994933022.py:8: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(path, scale=scale)


Saved plot to: step_2_output/Contributions_by_Institution.png


## Contributions by location (world map)

The same first-speaker counts placed on the globe; marker size = number of contributions.

In [ ]:
map_df = pd.DataFrame(locations).copy()
map_df["count"] = map_df["inst"].map(location_counts).fillna(0).astype(int)
map_df = map_df[map_df["count"] > 0].sort_values("count", ascending=False)

display(map_df[["inst", "city", "country", "count", "latitude", "longitude"]])

fig = px.scatter_geo(
    map_df,
    lat="latitude",
    lon="longitude",
    size="count",
    hover_name="label_short",
    hover_data={
        "institution": True,
        "city": True,
        "country": True,
        "species": True,
        "count": True,
        "latitude": False,
        "longitude": False,
    },
    color="species",
    projection="natural earth",
    title="Contributions by First Speaker Institution Location",
    labels={"count": "Contributions", "species": "Species"},
    size_max=30,
 )
fig.update_geos(showland=True, landcolor="rgb(243, 243, 243)", showcountries=True)
save_fig(fig, "Contribution_by_Location_Map.png")
fig.show()

,inst,city,country,count,latitude,longitude
8,DESY,Hamburg,Germany,14,53.5753,9.8828
38,CERN,Geneva,Switzerland,13,46.2044,6.0553
32,BNL,Upton,United States,9,40.8680,-72.8811
30,LBNL,Berkeley,United States,7,37.8755,-122.2477
2,FNAL,Batavia,United States,7,41.8419,-88.2543
5,SLAC,Menlo Park,United States,7,37.4529,-122.1817
40,ISIS,Didcot,United Kingdom,5,51.5734,-1.2442
46,PAL,Pohang,South Korea,4,36.0190,129.3435
14,JLab,Newport News,United States,3,37.0871,-76.4730
45,RSC,Sayo,Japan,2,34.8789,134.9946


Saved plot to: step_2_output/Contribution_by_Location_Map.png


/var/folders/n0/04pzpmy918x8zphf4zjqct2c0000j6/T/ipykernel_64766/1994933022.py:8: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(path, scale=scale)


## Institution collaboration network

### Build the collaboration graph

Count contributions per institution and tally **co-appearance edges**: two institutions get an edge each time they share a contribution (via speakers/authors/co-authors), weighted by how many they share.

In [ ]:
top_n_labels = 15

institution_contribution_counts = Counter()
edge_weights = Counter()

for contribution in data:
    person_to_institutions = {}
    unknown_index = 0

    for person in iter_people(contribution):
        inst = person.get("inst")
        if not inst:
            continue

        name = (person.get("name") or "").strip()
        if name:
            person_id = name
        else:
            unknown_index += 1
            person_id = f"<unknown-{unknown_index}>"

        person_to_institutions.setdefault(person_id, set()).add(inst)

    contribution_institutions = sorted(
        {inst for institutions in person_to_institutions.values() for inst in institutions}
    )

    for inst in contribution_institutions:
        institution_contribution_counts[inst] += 1

    for source_inst, target_inst in combinations(contribution_institutions, 2):
        edge_weights[(source_inst, target_inst)] += 1

locations_by_inst = {location["inst"]: location for location in locations}

base_node_rows = []
for inst, contribution_count in institution_contribution_counts.items():
    location = locations_by_inst.get(inst, {})
    base_node_rows.append(
        {
            "inst": inst,
            "label": location.get("label_short", inst),
            "institution": location.get("institution", inst),
            "country": location.get("country", "Unknown"),
            "species": location.get("species", "unknown"),
            "contributions": contribution_count,
        }
    )

base_node_df = pd.DataFrame(base_node_rows)

### Graph & layout helpers

`build_graph_data` keeps only institutions that share at least one edge, detects communities (greedy modularity), and lays each community out as its own ring cluster.

In [ ]:
def ring_cluster_layout(
    graph: nx.Graph,
    communities: list,
    ring_radius: float = 3.8,
    cluster_base_radius: float = 0.48,
    cluster_scale_per_node: float = 0.05,
):
    positions = {}
    community_centers = {}
    ordered_communities = sorted(communities, key=len, reverse=True)
    community_count = len(ordered_communities)

    for idx, community_nodes in enumerate(ordered_communities):
        nodes = list(community_nodes)
        nodes.sort(key=lambda node: graph.degree(node, weight="weight"), reverse=True)

        center_angle = 2 * np.pi * idx / max(community_count, 1)
        center = np.array([ring_radius * np.cos(center_angle), ring_radius * np.sin(center_angle)])
        community_centers[idx] = center

        node_count = len(nodes)
        if node_count == 1:
            positions[nodes[0]] = center + np.array([cluster_base_radius, 0.0])
            continue

        cluster_radius = cluster_base_radius + cluster_scale_per_node * min(node_count, 12)

        for node_index, node in enumerate(nodes):
            local_angle = center_angle + (2 * np.pi * node_index / node_count)
            offset = cluster_radius * np.array([np.cos(local_angle), np.sin(local_angle)])
            positions[node] = center + offset

    return positions, community_centers

def build_graph_data(min_edge_weight: int):
    edge_df = pd.DataFrame(
        [
            {"source": source, "target": target, "weight": weight}
            for (source, target), weight in edge_weights.items()
            if weight >= min_edge_weight
        ]
    )

    if edge_df.empty:
        raise ValueError(f"No collaboration edges for min_edge_weight={min_edge_weight}.")

    connected_institutions = set(edge_df["source"]).union(edge_df["target"])
    node_df = base_node_df[base_node_df["inst"].isin(connected_institutions)].copy()

    graph = nx.Graph()
    for row in node_df.to_dict("records"):
        graph.add_node(row["inst"], **row)
    for row in edge_df.to_dict("records"):
        graph.add_edge(row["source"], row["target"], weight=row["weight"])

    for inst, weighted_degree in graph.degree(weight="weight"):
        graph.nodes[inst]["weighted_degree"] = weighted_degree

    communities = list(nx.community.greedy_modularity_communities(graph, weight="weight"))
    community_by_inst = {}
    for community_index, community_nodes in enumerate(communities):
        for inst in community_nodes:
            community_by_inst[inst] = community_index

    ring_radius = max(3.0, 0.75 * len(communities))
    positions, community_centers = ring_cluster_layout(
        graph,
        communities,
        ring_radius=ring_radius,
        cluster_base_radius=0.5,
        cluster_scale_per_node=0.06,
    )

    node_plot_df = pd.DataFrame(
        [
            {
                **graph.nodes[inst],
                "x": float(positions[inst][0]),
                "y": float(positions[inst][1]),
                "community": community_by_inst.get(inst, -1),
                "community_cx": float(community_centers[community_by_inst.get(inst, -1)][0]),
                "community_cy": float(community_centers[community_by_inst.get(inst, -1)][1]),
            }
            for inst in graph.nodes
        ]
    ).sort_values(["weighted_degree", "contributions"], ascending=False)

    edge_df = edge_df.sort_values("weight", ascending=False)
    return node_plot_df, edge_df, positions

def make_edge_traces(edge_df: pd.DataFrame, positions: dict):
    traces = []
    max_edge_weight = edge_df["weight"].max()
    for row in edge_df.to_dict("records"):
        source_x, source_y = positions[row["source"]]
        target_x, target_y = positions[row["target"]]
        traces.append(
            go.Scatter(
                x=[source_x, target_x, None],
                y=[source_y, target_y, None],
                mode="lines",
                line={
                    "width": 1 + 5 * row["weight"] / max_edge_weight,
                    "color": "rgba(120, 120, 120, 0.35)",
                },
                hoverinfo="text",
                text=f"{row['source']} - {row['target']}: {row['weight']} shared contributions",
                showlegend=False,
            )
        )
    return traces

### Shared plotting helpers

In [ ]:
def add_common_layout(fig: go.Figure, title: str):
    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis={"visible": False},
        yaxis={"visible": False, "scaleanchor": "x", "scaleratio": 1},
        hovermode="closest",
        margin={"l": 20, "r": 20, "t": 50, "b": 20},
    )

def is_lab_institution(institution_name: str) -> bool:
    text = institution_name.lower()
    university_keywords = ("university", "universite", "universitat", "universidad")
    if any(keyword in text for keyword in university_keywords):
        return False

    lab_keywords = (
        "laboratory",
        "lab",
        "accelerator",
        "facility",
        "centre",
        "center",
        "synchrotron",
        "organization",
        "council",
        "institute",
        "helmholtz",
        "source",
        "riken",
    )
    return any(keyword in text for keyword in lab_keywords)

def outward_text_position(row: pd.Series) -> str:
    dx = row["x"] - row["community_cx"]
    dy = row["y"] - row["community_cy"]
    if abs(dx) >= abs(dy):
        return "middle right" if dx >= 0 else "middle left"
    return "top center" if dy >= 0 else "bottom center"

def top_connector_by_community(node_plot_df: pd.DataFrame, edge_df: pd.DataFrame) -> list[tuple[int, list[str]]]:
    community_by_inst = node_plot_df.set_index("inst")["community"].to_dict()
    weighted_degree_by_inst = node_plot_df.set_index("inst")["weighted_degree"].to_dict()
    intra_strength = Counter({inst: 0 for inst in node_plot_df["inst"].tolist()})

    for row in edge_df.itertuples(index=False):
        source = row.source
        target = row.target
        weight = row.weight
        if community_by_inst.get(source) == community_by_inst.get(target):
            intra_strength[source] += weight
            intra_strength[target] += weight

    connectors_by_community = []
    for community, group_df in node_plot_df.groupby("community"):
        candidates = group_df["inst"].tolist()
        ranked_connectors = sorted(
            candidates,
            key=lambda inst: (
                intra_strength.get(inst, 0),
                weighted_degree_by_inst.get(inst, 0),
                inst,
            ),
            reverse=True,
        )
        connectors_by_community.append((int(community), ranked_connectors[:3]))

    return sorted(connectors_by_community, key=lambda item: item[0])

def community_color(community: int, max_community: int) -> str:
    if max_community <= 0:
        return px.colors.sample_colorscale("Turbo", [0.5])[0]
    return px.colors.sample_colorscale("Turbo", [community / max_community])[0]

### Collaboration network — plotting function (colored by community)

In [ ]:
def make_community_plot(
    node_plot_df: pd.DataFrame,
    edge_df: pd.DataFrame,
    positions: dict,
    title: str,
    labels_mode: str = "top_n",
    legend_mode: str = "none",
    save_name: str | None = None,
):
    if labels_mode == "labs_only":
        show_label = node_plot_df["institution"].apply(is_lab_institution)
    else:
        top_label_set = set(node_plot_df.head(top_n_labels)["inst"])
        show_label = node_plot_df["inst"].isin(top_label_set)

    text_labels = [inst if keep else "" for inst, keep in zip(node_plot_df["inst"], show_label)]
    text_positions = node_plot_df.apply(outward_text_position, axis=1).tolist()

    node_trace = go.Scatter(
        x=node_plot_df["x"],
        y=node_plot_df["y"],
        mode="markers+text",
        text=text_labels,
        textposition=text_positions,
        hovertext=node_plot_df.apply(
            lambda row: (
                f"{row['institution']}<br>"
                f"Institution: {row['inst']}<br>"
                f"Country: {row['country']}<br>"
                f"Species: {row['species']}<br>"
                f"Contributions: {row['contributions']}<br>"
                f"Weighted degree: {row['weighted_degree']}<br>"
                f"Community: {row['community']}"
            ),
            axis=1,
        ),
        hoverinfo="text",
        marker={
            "size": 12 + node_plot_df["weighted_degree"],
            "color": node_plot_df["community"],
            "colorscale": "Turbo",
            "showscale": legend_mode != "community_top_connector",
            "colorbar": {"title": "Community"},
            "line": {"width": 1, "color": "white"},
            "opacity": 0.9,
        },
        showlegend=False,
    )

    fig = go.Figure(data=make_edge_traces(edge_df, positions) + [node_trace])

    if legend_mode == "community_top_connector":
        connectors_by_community = top_connector_by_community(node_plot_df, edge_df)
        max_community = int(node_plot_df["community"].max())
        for community, top_connectors in connectors_by_community:
            color = community_color(int(community), max_community)
            legend_label = " / ".join(top_connectors)
            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="markers",
                    marker={
                        "size": 10,
                        "color": color,
                        "line": {"width": 1, "color": "white"},
                    },
                    name=legend_label,
                    hoverinfo="skip",
                    showlegend=True,
                )
            )
        fig.update_layout(legend={"title": "Community Color Key"})
    add_common_layout(fig, title)
    if save_name:
        save_fig(fig, save_name)
    fig.show()

### Collaboration network — plotting function (colored by country)

In [ ]:
def country_to_continent(country: str) -> str:
    mapping = {
        "United States": "North America",
        "Canada": "North America",
        "Mexico": "North America",
        "Germany": "Europe",
        "France": "Europe",
        "Spain": "Europe",
        "Italy": "Europe",
        "Austria": "Europe",
        "Sweden": "Europe",
        "Switzerland": "Europe",
        "United Kingdom": "Europe",
        "Bosnia and Herzegovina": "Europe",
        "Japan": "Asia",
        "South Korea": "Asia",
        "China": "Asia",
        "India": "Asia",
        "Australia": "Oceania",
        "New Zealand": "Oceania",
        "Brazil": "South America",
        "Argentina": "South America",
        "Chile": "South America",
        "South Africa": "Africa",
        "Egypt": "Africa",
    }
    return mapping.get(country, "Other")

def build_country_color_map(node_plot_df: pd.DataFrame) -> dict:
    continent_scales = {
        "North America": px.colors.sequential.Oranges,
        "Europe": px.colors.sequential.Blues,
        "Asia": px.colors.sequential.Greens,
        "South America": px.colors.sequential.Reds,
        "Africa": px.colors.sequential.Purples,
        "Oceania": px.colors.sequential.PuBuGn,
        "Other": px.colors.sequential.Greys,
    }

    country_to_cont = {
        country: country_to_continent(country)
        for country in sorted(node_plot_df["country"].dropna().unique())
    }

    def assign_colors(countries: list[str], scale: list[str], color_map: dict):
        span_min = 0.25
        span_max = 0.75
        if not countries:
            return
        if len(countries) == 1:
            color_map[countries[0]] = scale[len(scale) // 2]
            return
        for idx, country in enumerate(sorted(countries)):
            pos = span_min + (span_max - span_min) * (idx / (len(countries) - 1))
            scale_idx = int(round(pos * (len(scale) - 1)))
            color_map[country] = scale[scale_idx]

    country_species_strength = (
        node_plot_df.groupby(["country", "species"]) ["contributions"]
        .sum()
        .reset_index()
    )

    dominant_species_by_country = {}
    for country, country_df in country_species_strength.groupby("country"):
        dominant_species_by_country[country] = country_df.sort_values("contributions", ascending=False).iloc[0]["species"]

    color_map = {}
    for continent, scale in continent_scales.items():
        countries = [c for c, cont in country_to_cont.items() if cont == continent]
        if not countries:
            continue

        if continent == "Europe":
            europe_electron = [
                c for c in countries if dominant_species_by_country.get(c, "") == "electrons"
            ]
            europe_other = [c for c in countries if c not in europe_electron]
            assign_colors(europe_electron, px.colors.sequential.RdPu, color_map)
            assign_colors(europe_other, px.colors.sequential.Blues, color_map)
            continue

        assign_colors(countries, scale, color_map)

    return color_map

def make_country_plot(node_plot_df: pd.DataFrame, edge_df: pd.DataFrame, positions: dict, title: str, save_name: str | None = None):
    fig = go.Figure(data=make_edge_traces(edge_df, positions))
    country_color_map = build_country_color_map(node_plot_df)

    country_species_strength = (
        node_plot_df.groupby(["country", "species"]) ["contributions"]
        .sum()
        .reset_index()
    )
    dominant_species_by_country = {}
    for country, country_df in country_species_strength.groupby("country"):
        dominant_species_by_country[country] = country_df.sort_values("contributions", ascending=False).iloc[0]["species"]

    continent_order = {
        "Africa": 0,
        "Asia": 1,
        "Europe": 2,
        "North America": 3,
        "South America": 4,
        "Oceania": 5,
        "Other": 6,
    }

    def country_sort_key(country: str):
        continent = country_to_continent(country)
        species = dominant_species_by_country.get(country, "unknown")
        if continent == "Europe":
            species_order = 0 if species == "hadrons" else 1
        else:
            species_order = 0
        return (continent_order.get(continent, 99), species_order, country)

    countries_in_order = sorted(node_plot_df["country"].dropna().unique(), key=country_sort_key)

    for country in countries_in_order:
        country_df = node_plot_df[node_plot_df["country"] == country]
        continent = country_to_continent(country)
        fig.add_trace(
            go.Scatter(
                x=country_df["x"],
                y=country_df["y"],
                mode="markers",
                name=country,
                marker={
                    "size": 10 + country_df["weighted_degree"],
                    "color": country_color_map.get(country, "#999999"),
                    "line": {"width": 0.5, "color": "white"},
                    "opacity": 0.9,
                },
                hovertext=country_df.apply(
                    lambda row: (
                        f"Country: {row['country']}<br>"
                        f"Continent: {continent}<br>"
                        f"Contributions: {row['contributions']}<br>"
                        f"Weighted degree: {row['weighted_degree']}"
                    ),
                    axis=1,
                ),
                hoverinfo="text",
            )
        )

    add_common_layout(fig, title)
    if save_name:
        save_fig(fig, save_name)
    fig.show()

### Plot: collaboration network by community

> Community **numbers/colors are not stable across runs** (community detection depends on dict ordering). The clustering is stable; only the labels may permute.

In [ ]:
node_df_step2, edge_df_step2, positions_step2 = build_graph_data(min_edge_weight=1)
display(node_df_step2.head(15))
display(edge_df_step2.head(20))
make_community_plot(
    node_df_step2,
    edge_df_step2,
    positions_step2,
    title="Institution Collaboration Network (Labs Labeled)",
    labels_mode="labs_only",
    legend_mode="community_top_connector",
    save_name="Institution_Collaboration_Network.png",
)

,inst,label,institution,country,species,contributions,weighted_degree,x,y,community,community_cx,community_cy
4,JLab,"JLab, Newport News, USA",Thomas Jefferson National Accelerator Facility,United States,electrons,10,22,4.935605e+00,4.935605,1,4.242641e+00,4.242641
0,DESY,"DESY, Hamburg, DEU",Deutsches Elektronen-Synchrotron,Germany,electrons,17,21,7.220000e+00,0.000000,0,6.000000e+00,0.000000
2,SLAC,"SLAC, Menlo Park, USA",SLAC National Accelerator Laboratory,United States,electrons,13,20,7.080256e+00,0.566962,0,6.000000e+00,0.000000
5,LBNL,"LBNL, Berkeley, USA",Lawrence Berkeley National Laboratory,United States,electrons,9,20,6.693039e+00,1.004040,0,6.000000e+00,0.000000
3,BNL,"BNL, Upton, USA",Brookhaven National Laboratory,United States,hadrons,11,17,4.242641e+00,5.222641,1,4.242641e+00,4.242641
11,ANL,"ANL, Lemont, USA",Argonne National Laboratory,United States,hadrons,4,14,6.147055e+00,1.211105,0,6.000000e+00,0.000000
25,TUHH,"TUHH, Hamburg, DEU",Hamburg University of Technology,Germany,electrons,4,11,5.567382e+00,1.140720,0,6.000000e+00,0.000000
6,Cornell,"Cornell, Ithaca, USA",Cornell University,United States,hadrons,6,9,3.262641e+00,4.242641,1,4.242641e+00,4.242641
13,ORNL,"ORNL, Oak Ridge, USA",Oak Ridge National Laboratory,United States,hadrons,3,9,3.549676e+00,4.935605,1,4.242641e+00,4.242641
23,ASTeC,"ASTeC, Daresbury, GBR",Accelerator Science and Technology Centre,United Kingdom,electrons,2,8,4.815451e+00,-0.291965,0,6.000000e+00,0.000000


,source,target,weight
18,BNL,JLab,5
6,BNL,Cornell,5
29,LBNL,SLAC,4
57,DESY,TUHH,4
34,Cornell,JLab,3
26,JLab,ORNL,3
4,DESY,LBNL,3
5,JLab,LBNL,3
16,ANL,SLAC,2
76,RIKEN,RNC,2


Saved plot to: step_2_output/Institution_Collaboration_Network.png


/var/folders/n0/04pzpmy918x8zphf4zjqct2c0000j6/T/ipykernel_64766/1994933022.py:8: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(path, scale=scale)


### Plot: collaboration network by country

In [ ]:
make_country_plot(
    node_df_step2,
    edge_df_step2,
    positions_step2,
    title="Institution Collaboration Network by Country",
    save_name="Institution_Collaboration_Network_by_Country.png",
)

Saved plot to: step_2_output/Institution_Collaboration_Network_by_Country.png


/var/folders/n0/04pzpmy918x8zphf4zjqct2c0000j6/T/ipykernel_64766/1994933022.py:8: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(path, scale=scale)


### Institutions by community — text report

Writes `step_2_output/institution_submissions_by_community.txt` (regenerated each run).

In [ ]:
# Print institutions by community with submission numbers
# Build mapping of institutions to submission IDs
inst_to_submission_ids = {inst: [] for inst in node_df_step2["inst"].tolist()}

for submission_id, contribution in enumerate(data):
    fs = first_speaker(contribution)
    if fs is None:
        continue
    
    first_speaker_inst = fs.get("inst")
    if first_speaker_inst and first_speaker_inst in inst_to_submission_ids:
        inst_to_submission_ids[first_speaker_inst].append(submission_id)

# Create mapping of institutions to communities
inst_to_community = node_df_step2.set_index("inst")["community"].to_dict()

community_lines = ["=== Institutions by Community (with Submission Numbers) ==="]
for community_id in sorted(node_df_step2["community"].unique()):
    community_df = node_df_step2[node_df_step2["community"] == community_id].sort_values("contributions", ascending=False)
    community_lines.append(f"\nCommunity {community_id}:")
    for _, row in community_df.iterrows():
        submission_ids = inst_to_submission_ids.get(row['inst'], [])
        submission_str = ", ".join(map(str, sorted(submission_ids)))
        community_lines.append(f"  {row['inst']}: {row['contributions']} contributions (submissions: {submission_str})")

community_report = "\n".join(community_lines)
print("\n" + community_report)

# Also save the same listing as a text artifact alongside the other step-2 outputs.
community_report_path = output_dir / "institution_submissions_by_community.txt"
with community_report_path.open("w", encoding="utf-8") as f:
    f.write(community_report + "\n")
print(f"\nSaved community report to: {community_report_path}")


=== Institutions by Community (with Submission Numbers) ===

Community 0:
  DESY: 17 contributions (submissions: 0, 1, 4, 10, 31, 33, 34, 53, 54, 55, 56, 64, 67, 70)
  SLAC: 13 contributions (submissions: 3, 12, 51, 57, 58, 61, 93)
  LBNL: 9 contributions (submissions: 8, 9, 24, 41, 44, 50, 60)
  ISIS: 5 contributions (submissions: 66, 69, 75, 76, 81)
  ANL: 4 contributions (submissions: 29, 59)
  TUHH: 4 contributions (submissions: 82)
  ASTeC: 2 contributions (submissions: 49)
  LU: 2 contributions (submissions: 26)
  KIT: 1 contributions (submissions: )
  UChicago: 1 contributions (submissions: )
  UoL: 1 contributions (submissions: )
  HZB: 1 contributions (submissions: )
  UniSarajevo: 1 contributions (submissions: )

Community 1:
  BNL: 11 contributions (submissions: 2, 11, 25, 30, 36, 40, 43, 72, 73)
  JLab: 10 contributions (submissions: 27, 32, 35)
  FNAL: 9 contributions (submissions: 47, 48, 79, 80, 85, 89, 91)
  Cornell: 6 contributions (submissions: 7)
  ORNL: 3 contribut

### Regenerate this folder's `README.md`

Auto-builds `step_2_output/README.md` (plot gallery + community table) so it stays in sync with the outputs above.

In [ ]:
# Regenerate step_2_output/README.md: a gallery of the plots + the community
# table. Rebuilt on every run from the files this notebook writes here; the
# gallery assumes these .png file names exist in this directory.
plots = [
    ("Contributions_by_Institution.png",
     "Contributions by first-speaker institution",
     "Contributions per institution (attributed to the first speaker), colored by "
     "species — electron- vs hadron-machine institutions."),
    ("Contribution_by_Location_Map.png",
     "Contributions by location",
     "The same first-speaker counts placed on a world map; marker size is the "
     "number of contributions."),
    ("Institution_Collaboration_Network.png",
     "Institution collaboration network — by community",
     "Institutions are linked whenever they co-appear on a contribution (weighted "
     "by how many they share); nodes are colored by detected community."),
    ("Institution_Collaboration_Network_by_Country.png",
     "Institution collaboration network — by country",
     "The same collaboration network, colored by country instead of community."),
]

lines = [
    "# Step 2 outputs",
    "",
    "<!-- AUTO-GENERATED by step_2_explore_contributions.ipynb on every run. -->",
    "<!-- Do not edit by hand. The gallery assumes the .png file names below are -->",
    "<!-- the ones this notebook writes into this directory. -->",
    "",
    "Rendered snapshots and the community listing produced by",
    "`step_2_explore_contributions.ipynb`. Regenerate with `uv run main.py` (or by",
    "running the notebook top to bottom).",
    "",
    "> **Snapshot of one run.** Community detection is not deterministic, so the",
    "> community numbers/colors may permute if you re-run — the clustering is stable.",
    ">",
    "> **One institution per person.** Where an individual listed more than one",
    "> institution, only one was chosen, so each person counts toward a single",
    "> institution.",
    "",
    "## Plots",
    "",
]
for fname, title, desc in plots:
    lines += [f"### {title}", "", desc, "", f"![{title}]({fname})", ""]

lines += [
    "## Institutions by community",
    "",
    "**Contributions** counts every contribution an institution is credited on "
    "(speaker, author, or co-author); **first-speaker submissions** lists the "
    "contributions where that institution provided the first speaker (`—` = none).",
    "",
    "| Community | Institution | Contributions | First-speaker submissions |",
    "| ---: | --- | ---: | --- |",
]
for community_id in sorted(node_df_step2["community"].unique()):
    community_df = node_df_step2[node_df_step2["community"] == community_id].sort_values(
        "contributions", ascending=False
    )
    first = True
    for _, row in community_df.iterrows():
        subs = inst_to_submission_ids.get(row["inst"], [])
        subs_str = ", ".join(map(str, sorted(subs))) if subs else "—"
        lines.append(f"| {community_id if first else ''} | {row['inst']} | {row['contributions']} | {subs_str} |")
        first = False

readme_path = output_dir / "README.md"
with readme_path.open("w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print(f"Wrote gallery README to: {readme_path}")

Wrote gallery README to: step_2_output/README.md
